# N-step DQN: construct multi-step replay targets

N-step DQN replaces a one-step reward with
$$G_t^{(n)}=\sum_{k=0}^{n-1}\gamma^kR_{t+k+1}+\gamma^n(1-d)\max_aQ_{\theta^-}(S_{t+n},a).$$
Here $n$ is the horizon, $\gamma$ the discount, $R$ sampled rewards, $d$ the terminal indicator, and $Q_{\theta^-}$ the target network. This notebook isolates the bookkeeping that converts one-step transitions into n-step replay items.

## 1. Accumulate transitions

A short deque delays insertion until enough rewards are available. At an episode boundary, every remaining prefix is flushed.

In [ ]:
from collections import deque
import matplotlib.pyplot as plt
import numpy as np

N_STEPS = 3
GAMMA = 0.99
pending = deque()
replay = []

def emit(force=False):
    while pending and (len(pending) >= N_STEPS or force):
        horizon = min(N_STEPS, len(pending))
        reward = sum(GAMMA**k * pending[k][2] for k in range(horizon))
        state, action = pending[0][:2]
        next_state, terminal = pending[horizon - 1][3:]
        replay.append((state, action, reward, next_state, terminal, GAMMA**horizon))
        pending.popleft()
        if not force:
            break

def add_transition(state, action, reward, next_state, terminal, episode_end):
    pending.append((state, action, reward, next_state, terminal))
    emit()
    if episode_end:
        emit(force=True)

## 2. Verify the exact return

A deterministic reward sequence makes every stored reward and bootstrap discount inspectable.

In [ ]:
rewards = [1.0, 2.0, 3.0, 4.0]
for step, reward in enumerate(rewards):
    add_transition(step, 0, reward, step + 1, terminal=step == 3, episode_end=step == 3)
for item in replay:
    print(item)
assert np.isclose(replay[0][2], 1 + GAMMA * 2 + GAMMA**2 * 3)

## 3. Visualize horizon effects

Longer horizons incorporate more observed reward before bootstrapping.

In [ ]:
horizons = np.arange(1, len(rewards) + 1)
returns = [sum(GAMMA**k * rewards[k] for k in range(h)) for h in horizons]
plt.plot(horizons, returns, marker="o")
plt.xlabel("Return horizon")
plt.ylabel("Discounted sampled reward")
plt.title("N-step return before bootstrapping")
plt.grid(alpha=0.2)
plt.show()